
# Edge AI Assignment: Neural Network from Scratch (Checkerboard Classification)

## Objective
Build a simple neural network **from scratch (no ML libraries)** to classify a 2D checkerboard pattern.

You will:
- Implement forward propagation
- Implement backpropagation
- Train a small neural network
- Visualize the decision boundary

---

## Dataset Concept
We generate (x, y) points and assign labels based on a checkerboard pattern.

This is a **non-linear classification problem** (like XOR, but extended to 2D).


In [ ]:

import numpy as np
import matplotlib.pyplot as plt


## Step 1: Generate Checkerboard Data

In [ ]:

def generate_data(n_samples=500, k=2):
    X = np.random.uniform(-1, 1, (n_samples, 2))

    # Checkerboard labels
    y = ((np.floor((X[:,0]+1)*k) + np.floor((X[:,1]+1)*k)) % 2)

    return X, y.reshape(-1, 1)

X, y = generate_data()


## Visualize Dataset

In [ ]:

plt.scatter(X[:,0], X[:,1], c=y.flatten())
plt.title("Checkerboard Dataset")
plt.show()



## Step 2: Build Neural Network

Architecture:
- Input: 2 neurons (x, y)
- Hidden: 32 neurons with **tanh** activation
- Output: 1 neuron with **sigmoid** activation

**Why tanh in the hidden layer?**
Sigmoid in hidden layers suffers from vanishing gradients (its derivative
maxes out at 0.25 and shrinks fast as activations saturate). Tanh is
centered at zero and has a stronger derivative (max 1.0), so gradients
flow much better through deeper networks. We keep sigmoid at the output
because we need a probability in (0, 1) for binary classification.


In [ ]:

# Initialize weights
def init_params(input_size, hidden_size, output_size):
    """Initialize weights with Xavier-style scaling.
    
    Naive randn() initialization gives weights too large for tanh,
    causing immediate saturation. Scaling by 1/sqrt(fan_in) keeps
    pre-activations in tanh's linear region at the start, which lets
    gradients flow well in early epochs.
    """
    W1 = np.random.randn(input_size, hidden_size) * np.sqrt(1.0 / input_size)
    b1 = np.zeros((1, hidden_size))
    
    W2 = np.random.randn(hidden_size, output_size) * np.sqrt(1.0 / hidden_size)
    b2 = np.zeros((1, output_size))
    
    return W1, b1, W2, b2


## Activation Functions

In [ ]:

def sigmoid(x):
    """Squash any real number into (0, 1)."""
    # Clip to prevent overflow on very large negative inputs
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def sigmoid_derivative(x):
    """d/dx sigmoid(x) = sigmoid(x) * (1 - sigmoid(x)).
    
    Used for the OUTPUT layer's gradient (when not paired with BCE
    where it cancels out — see backward function below).
    """
    s = sigmoid(x)
    return s * (1 - s)

def tanh_derivative(x):
    """d/dx tanh(x) = 1 - tanh^2(x). Used for the HIDDEN layer."""
    return 1 - np.tanh(x) ** 2


## Forward Pass

In [ ]:

def forward(X, W1, b1, W2, b2):
    # 1. Hidden layer linear combination
    Z1 = X @ W1 + b1
    
    # 2. Activation
    A1 = np.tanh(Z1)          # tanh activation in hidden layer
    
    # 3. Output layer
    Z2 = A1 @ W2 + b2
    A2 = sigmoid(Z2)          # sigmoid for probability output
    
    # 4. Return output + intermediates
    cache = (Z1, A1, Z2, A2)
    return A2, cache


## Loss Function

$$L = -\frac{1}{m} \sum_i \left[ y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \right]$$

The clip on predictions prevents `log(0)` from blowing up to infinity
when the network gets very confident.

In [ ]:

def compute_loss(y_true, y_pred):
    # Binary cross-entropy
    m = y_true.shape[0]
    y_pred = np.clip(y_pred, 1e-9, 1 - 1e-9)  # prevent log(0)
    loss = -np.sum(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)) / m
    return loss


## Backpropagation

In [ ]:

def backward(X, y, W1, b1, W2, b2, cache):
    # TODO:
    # Compute gradients for all parameters
    Z1, A1, Z2, A2 = cache
    m = y.shape[0]

    # Output layer gradients (BCE + sigmoid simplifies to A2 - y)
    dZ2 = A2 - y                                      # (m, 1)
    dW2 = A1.T @ dZ2 / m                              # (hidden, 1)
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m      # (1, 1)

    # Hidden layer gradients
    dA1 = dZ2 @ W2.T                                  # (m, hidden)
    dZ1 = dA1 * tanh_derivative(Z1)                   # (m, hidden)
    dW1 = X.T @ dZ1 / m                               # (input, hidden)
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m      # (1, hidden)

    return dW1, db1, dW2, db2


## Training Loop

In [ ]:

def train(X, y, hidden_size=8, lr=0.1, epochs=1000, log_every=1000):
    W1, b1, W2, b2 = init_params(2, hidden_size, 1)
    history = {'loss': [], 'accuracy': []}
    
    for epoch in range(epochs):
        # Forward
        A2, cache = forward(X, W1, b1, W2, b2)

        # Loss
        loss = compute_loss(y, A2)

        preds = (A2 > 0.5).astype(int)
        accuracy = np.mean(preds == y)
        history['loss'].append(loss)
        history['accuracy'].append(accuracy)
    
        # Backward
        dW1, db1, dW2, db2 = backward(X, y, W1, b1, W2, b2, cache)

        # Update weights
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

        if epoch % log_every == 0:
            print(f"Epoch {epoch} | Loss: {loss:.4f} | accuracy = {accuracy:.3f}")

    # Final stats
    A2, _ = forward(X, W1, b1, W2, b2)
    final_loss = compute_loss(y, A2)
    final_acc = np.mean((A2 > 0.5).astype(int) == y)
    print(f"\nFinal     | loss = {final_loss:.4f} | accuracy = {final_acc:.3f}")
    
    return W1, b1, W2, b2, history


## Visualize Decision Boundary

In [ ]:

def plot_decision_boundary(X, y, model, k=2):
    W1, b1, W2, b2 = model
    
    xx, yy = np.meshgrid(np.linspace(-1,1,100), np.linspace(-1,1,100))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # run forward pass on grid
    Z, _ = forward(grid, W1, b1, W2, b2)

    Z = Z.reshape(xx.shape)

    # Side-by-side: predicted vs ground truth
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Left: network's predictions
    axes[0].contourf(xx, yy, Z, alpha=0.5)
    axes[0].scatter(X[:, 0], X[:, 1], c=y.flatten())
    axes[0].set_title('Network Predictions')
    
    # Right: ground truth checkerboard
    truth = ((np.floor((xx + 1) * k) + np.floor((yy + 1) * k)) % 2)
    axes[1].contourf(xx, yy, truth, alpha=0.5)
    axes[1].scatter(X[:, 0], X[:, 1], c=y.flatten())
    axes[1].set_title('Ground Truth')
    
    plt.show()


## Run Training

In [ ]:

# Train your model
W1, b1, W2, b2, history = train(X, y, hidden_size=32, lr=0.5, epochs=15000)
model = (W1, b1, W2, b2)

plot_decision_boundary(X, y, model, k=2)

# Plot loss curve
plt.plot(history['loss'])
plt.title('Loss over epochs')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.show()


---

## ⭐ Extra Credit ( For Concept And Code, 20% add on )

### Circle Classification Problem

Instead of a checkerboard, classify points based on whether they lie inside a circle:

- Input: (x, y)
- Output: 1 if inside circle, else 0

Decision boundary:
x² + y² < r²

### Questions:
1. Why is this non-linear?
2. Would a single-layer perceptron work?
3. How would the decision boundary differ from the checkerboard?

### Answers:
1. A straight line cannot enclose a circular region. Inside-vs-outside-a-circle requires a curved boundary
2. No. A perceptron can only draw straight lines. It would fail completely on a circle.
3. The circle has ONE smooth closed curve as boundary. The checkerboard has multiple piecewise boundaries - much harder pattern to learn.

### Implementation

Try training the same network on circle data:




In [ ]:
def generate_circle_data(n_samples=500, radius=0.5):
    """Points inside circle of given radius are class 1, outside are class 0."""
    X = np.random.uniform(-1, 1, (n_samples, 2))
    y = (X[:, 0]**2 + X[:, 1]**2 < radius**2).astype(int)
    return X, y.reshape(-1, 1)

# Generate and visualize
np.random.seed(42)
X_circ, y_circ = generate_circle_data(n_samples=500, radius=0.5)

# Train on circle (much easier problem, fewer epochs)
W1c, b1c, W2c, b2c, _ = train(X_circ, y_circ, hidden_size=16, lr=0.5, epochs=5000)

# Visualize decision boundary
xx, yy = np.meshgrid(np.linspace(-1, 1, 100), np.linspace(-1, 1, 100))
grid = np.c_[xx.ravel(), yy.ravel()]
Z, _ = forward(grid, W1c, b1c, W2c, b2c)
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.5)
plt.scatter(X_circ[:, 0], X_circ[:, 1], c=y_circ.flatten())
plt.title('Circle Classification')
plt.show()